In [ ]:
%matplotlib inline
import pickle
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt

import os as _os, sys as _sys
_HERE = _os.getcwd()
_ROOT = _HERE if _os.path.isdir(_os.path.join(_HERE, "mcpilco")) else _os.path.dirname(_HERE)
_os.chdir(_ROOT)
_sys.path.insert(0, _ROOT)
_sys.path.insert(0, _os.path.dirname(_ROOT))

from mcpilco.config_single_phase import get_config
from mcpilco.pensim_wrapper import (PenSimWrapper, PenSimMCPILCO, STATE_NAMES,
                                     STATE_RANGES, STATE_DIM, WARMUP_H, T_SAMPLING)

# The RL action is the Fs (sugar-feed) residual, so this diagnostic focuses on the
# state dims Fs actually drives -- biomass X and penicillin P -- not PAA.
P_IDX = STATE_NAMES.index("P")
X_IDX = STATE_NAMES.index("X")
ACTION_IDX = STATE_DIM          # Fs-residual column in gp_inputs = [state(STATE_DIM), action]

In [ ]:
seed = 0
trial = None
num_trials = 10
fast = False
results_dir = "results/single_phase/seed0_0/"

In [ ]:
def _denorm(x, lo, hi):
    return lo + (x + 1.0) * (hi - lo) / 2.0


def _denorm_delta(d, lo, hi):
    return d * (hi - lo) / 2.0


def _resolve_trial(log, trial):
    avail = sorted(int(k.split("_")[-1]) for k in log if k.startswith("parameters_gp_"))
    if not avail:
        raise RuntimeError("log.pkl has no parameters_gp_<i> (no trained GP to load)")
    if trial is None:
        return avail[-1]
    if trial not in avail:
        raise RuntimeError(f"trial {trial} not in saved GP trials {avail}")
    return trial


def reconstruct(seed, num_trials, fast, log, idx):
    cfg = get_config(seed=seed, num_trials=num_trials, fast=fast)
    cfg["mc_pilco_init"]["log_path"] = None
    agent = PenSimMCPILCO(pensim_wrapper=PenSimWrapper(**cfg["wrapper_par"]),
                          **cfg["mc_pilco_init"])
    agent.state_samples_history = log["state_samples_history"]
    agent.input_samples_history = log["input_samples_history"]
    agent.noiseless_states_history = log.get("noiseless_states_history",
                                             log["state_samples_history"])

    ml = agent.model_learning
    ml.gp_inputs = log[f"gp_inputs_{idx}"]
    ml.gp_output_list = log[f"gp_output_list_{idx}"]
    ml.num_samples = ml.gp_inputs.shape[0]
    ml.dim_state = len(STATE_NAMES)
    ml.init_gp_models()
    params = log[f"parameters_gp_{idx}"]
    for k in range(ml.num_gp):
        ml.gp_list[k].load_state_dict(params[k])
        ml.norm_list[k] = torch.max(torch.abs(ml.gp_output_list[k]))
    with torch.no_grad():
        for k in range(ml.num_gp):
            ml.pretrain_gp(k)
    ml.set_eval_mode()
    return agent

In [ ]:
# d = Path(results_dir) / f"seed{seed}"
d = Path(results_dir)
log = pickle.load(open(d / "log.pkl", "rb"))
idx = _resolve_trial(log, trial)

agent = reconstruct(seed, num_trials, fast, log, idx)

with torch.no_grad():
    _, targets, means, _ = agent.get_model_learning_performance(idx)
    pred, true, _ = agent.get_rollout_prediction_performance(idx)

In [ ]:
per_dim_mse = [float(((targets[k] - means[k]) ** 2).mean()) for k in range(len(targets))]

# One-step biomass delta (dX): the state Fs most directly drives.
lo, hi = STATE_RANGES["X"]
tgt_dx = _denorm_delta(targets[X_IDX].ravel(), lo, hi)
prd_dx = _denorm_delta(means[X_IDX].ravel(), lo, hi)
ss_res = float(((tgt_dx - prd_dx) ** 2).sum())
ss_tot = float(((tgt_dx - tgt_dx.mean()) ** 2).sum()) or 1.0
r2_x = 1.0 - ss_res / ss_tot

# colour the scatter by the Fs action (residual in [-1, 1]) applied at each step,
# to see whether the GP captures the action -> dynamics effect.
gp_inputs = agent.model_learning.data_to_gp_input(
    torch.tensor(agent.state_samples_history[idx]),
    torch.tensor(agent.input_samples_history[idx]))[:-1, :].detach().cpu().numpy()
action_col = gp_inputs[:, ACTION_IDX]

t = WARMUP_H + np.arange(pred.shape[0]) * T_SAMPLING
x_pred = _denorm(pred[:, X_IDX], lo, hi);  x_true = _denorm(true[:, X_IDX], lo, hi)
p_pred = _denorm(pred[:, P_IDX], *STATE_RANGES["P"])
p_true = _denorm(true[:, P_IDX], *STATE_RANGES["P"])

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(14, 10))

sc = ax[0, 0].scatter(tgt_dx, prd_dx, c=action_col, cmap="viridis", s=14, alpha=.8)
lim = [min(tgt_dx.min(), prd_dx.min()), max(tgt_dx.max(), prd_dx.max())]
ax[0, 0].plot(lim, lim, "r--", lw=1.5, label="perfect (y=x)")
fig.colorbar(sc, ax=ax[0, 0], label="Fs action (residual -1..+1)")
ax[0, 0].set_title(f"One-step dX: GP vs actual  (R^2={r2_x:.3f})")
ax[0, 0].set_xlabel("actual dX (g/L per step)")
ax[0, 0].set_ylabel("GP predicted dX (g/L per step)")
ax[0, 0].grid(alpha=.3); ax[0, 0].legend(fontsize=8)

ax[0, 1].bar(range(len(per_dim_mse)), per_dim_mse, color="steelblue")
ax[0, 1].bar([X_IDX], [per_dim_mse[X_IDX]], color="crimson", label="X")
ax[0, 1].set_xticks(range(len(STATE_NAMES))); ax[0, 1].set_xticklabels(STATE_NAMES, rotation=45)
ax[0, 1].set_title("Per-dim one-step MSE (normalised delta)")
ax[0, 1].set_ylabel("MSE"); ax[0, 1].grid(alpha=.3, axis="y"); ax[0, 1].legend(fontsize=8)

ax[1, 0].plot(t, x_true, "k-", lw=2, label="actual (simulator)")
ax[1, 0].plot(t, x_pred, "C1--", lw=2, label="GP rollout")
ax[1, 0].set_title("Multi-step X (biomass): GP rollout vs simulator")
ax[1, 0].set_xlabel("time (h)"); ax[1, 0].set_ylabel("X (g/L)"); ax[1, 0].grid(alpha=.3)
try:
    mon = pickle.load(open(d / "monitor.pkl", "rb"))[idx]
    axt = ax[1, 0].twinx()
    axt.plot(mon["t"], mon["Fs"], color="purple", lw=1, alpha=.5, label="Fs (action)")
    axt.set_ylabel("Fs (L/h)", color="purple")
    axt.set_xlim(t[0], t[-1])
except (FileNotFoundError, IndexError, KeyError):
    pass
ax[1, 0].legend(fontsize=8, loc="upper left")

ax[1, 1].plot(t, p_true, "k-", lw=2, label="actual (simulator)")
ax[1, 1].plot(t, p_pred, "C1--", lw=2, label="GP rollout")
ax[1, 1].set_title("Multi-step P: GP rollout vs simulator")
ax[1, 1].set_xlabel("time (h)"); ax[1, 1].set_ylabel("P (g/L)")
ax[1, 1].grid(alpha=.3); ax[1, 1].legend(fontsize=8)

fig.suptitle(f"GP-vs-simulator diagnostic — seed {seed}, trial {idx}")
fig.tight_layout()
plt.show()